# 016 面板 ICE 验证（400 分子，Colab T4，仅跑 iceberg 预测）
按顺序跑，每格末尾有 EXPECT，对不上就停、把日志贴回来。FP 前向 + 窗口 + top100 已在本地 CPU 完成（v10 冻结口径），本 notebook 只跑 iceberg `split/k=100` 预测。
truth 仅最终评分用，worker 不读。400 分子预计 60–90 分钟；`cache/` 命中可断点续跑（重跑同格即可），保持页面在线。
Drive 上传 3 个文件：`panel-016-ice-bundle.tar.gz` + `gen.pt` + `inten.pt`（放 `MyDrive/016-panel/` 或 Drive 根，任选其一，第 2 格自动搜）。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# EXPECT: Mounted at /content/drive


In [ ]:
import os
SEARCH_DIRS = ['/content/drive/MyDrive/016-panel', '/content/drive/MyDrive', '/content']
NEED = ['panel-016-ice-bundle.tar.gz', 'gen.pt', 'inten.pt']
found = {}
for name in NEED:
    hit = next((d + '/' + name for d in SEARCH_DIRS if os.path.isfile(d + '/' + name)), None)
    found[name] = hit
    print(name, '->', hit if hit else 'NOT FOUND')
    assert hit, 'missing ' + name
import json as _j
print('EXPECT: 3 files all found')


In [ ]:
import os, shutil, hashlib
shutil.rmtree('/content/panel-016', ignore_errors=True)
tar = '/content/drive/MyDrive/016-panel/panel-016-ice-bundle.tar.gz'
import os as _o
for _d in SEARCH_DIRS:
    _p = _d + '/panel-016-ice-bundle.tar.gz'
    if _o.path.isfile(_p): tar = _p; break
!tar xzf "{tar}" -C /content && test -f /content/panel-016/inputs.json && ls /content/panel-016
# EXPECT: 看到 inputs.json MANIFEST.json f1_worker.py install.sh code/
h = hashlib.sha256(open('/content/panel-016/inputs.json','rb').read()).hexdigest()
man = _j.load(open('/content/panel-016/MANIFEST.json'))
print('inputs.json sha:', h[:16], 'molecules:', man.get('stat', {}))


In [ ]:
import shutil, os
os.makedirs('/content/panel-016/weights', exist_ok=True)
import os as _o
for _n in ['gen.pt', 'inten.pt']:
    _hit = next(_d + '/' + _n for _d in SEARCH_DIRS if _o.path.isfile(_d + '/' + _n))
    shutil.copy(_hit, '/content/panel-016/weights/' + _n)
!ls -l /content/panel-016/weights/
# EXPECT: gen.pt 约 14MB，inten.pt 约 13MB


In [ ]:
import torch
assert torch.cuda.is_available(), '本格必须 T4 GPU 运行时'
print(torch.cuda.get_device_name(0))
print('torch', torch.__version__)
# EXPECT: Tesla T4


In [ ]:
!cd /content/panel-016 && bash install.sh
# EXPECT: 末尾 install done. 约 2-4 分钟。注意 --any-rdkit 语义：PyPI rdkit 版本见输出 meta，非面板精确版


In [ ]:
!cd /content/panel-016 && PYTHONPATH=code/ice/shim:code/ice/src python3 f1_worker.py --pkg code/ice --model iceberg --inputs inputs.json --out /tmp/dry.json --k 100 --cemode split --dry-run --mol-limit 2
# EXPECT: status ok-dry-run，无 Traceback。只验接线，不跑模型。


In [ ]:
!cd /content/panel-016 && mkdir -p scores cache work work/ice_ckpt_pkg/ckpt && ln -sf $(pwd)/weights/gen.pt work/ice_ckpt_pkg/ckpt/gen.pt && ln -sf $(pwd)/weights/inten.pt work/ice_ckpt_pkg/ckpt/inten.pt && PYTHONPATH=code/ice/shim:code/ice/src python3 f1_worker.py --pkg code/ice --model iceberg --inputs inputs.json --out scores/panel016_ice_top100_split.json --k 100 --cemode split --device cuda --cache-dir cache --workdir work --ckpt-pkg work/ice_ckpt_pkg --any-rdkit
# EXPECT: status=ok。60-90 分钟；中途掉线重跑本格即可（cache 续跑）。


In [ ]:
!cd /content/panel-016 && ls -lh scores/ && tar czf panel016-ice-scores.tar.gz scores && ls -lh panel016-ice-scores.tar.gz
# EXPECT: scores/ 下有 panel016_ice_top100_split.json(+.meta.json)；去左侧 Files 面板下载 panel016-ice-scores.tar.gz 发回。
